# 04 - Output preflight and output control

This notebook isolates output discovery, selection, and preflight validation so output schemas are stable before heavy runs.

In practical terms, we will move through preparation, configuration, execution, and verification so every transition in the workflow stays explicit.
As the notebook progresses, each code cell is introduced with context first, then the call itself, and finally quick guidance on how to interpret the result.

Workflow roadmap:
- Preparation: imports, path configuration, and baseline model loading.
- Configuration: output selection, parameter definitions, and problem setup.
- Execution and review: simulation or optimization calls followed by targeted checks.

Main classes, methods, and functions used in this notebook:
- `ParametricSimulation.discover_available_outputs`
- `ParametricSimulation.select_outputs`
- `ParametricSimulation.apply_outputs_preflight`
- `ParametricSimulation.set_output_readers`
- `ef.get_building`
- `ParametricSimulation`
- `parametric.scan_output_objects`
- `parametric.autocorrect_output_duplicates`
- `parametric.discover_available_outputs`
- `parametric.select_outputs`

Narrative structure used in this notebook:
- Each code cell is preceded by an explanatory markdown block.
- Section headings are contextual to the current notebook and execution block.
- Key arguments are described before they are used in code.
- Execution-oriented cells may include a follow-up markdown note to help interpret results.

## Output preflight: import workflow dependencies

We start by importing the APIs required for this notebook block.

Keeping imports explicit at the top of each topic notebook makes selective reruns safer when you jump between sections of the series.

What to verify right after running this cell:
- Review execution output and proceed when values look consistent with the workflow intent.

In [1]:
from pathlib import Path

import accim
from besos import eppy_funcs as ef
from accim.parametric_and_optimisation.main import ParametricSimulation


## Output preflight: load the baseline IDF model

Load the base building model used by subsequent runs.

This is where the baseline IDF is loaded for all downstream operations. Later cells reuse this object for output setup, parameterization, and execution.

`reduce_runtime` also changes the run period, to 1 January by default, so its run period arguments keep the full year: the results stay annual. `Run_Simulation_for_Sizing_Periods = "No"` keeps the design days of the sample model out of the annual results (they are still used to size the HVAC system).

Key arguments used in this cell and why they matter:
- `idf_object`: defines this setting for the current call.
- `timesteps`: defines this setting for the current call.
- `building`: loaded IDF model used in this workflow.
- `parameters_type`: parameter family used in the simulation setup.
- `epws`: list of EPW weather files used to run scenarios.
- `output_freqs`: time frequency requested for outputs.

Variables created in this cell:
- `PROJECT_ROOT`: reused in subsequent workflow blocks.
- `idf_file`: reused in subsequent workflow blocks.
- `epw_list`: reused in subsequent workflow blocks.
- `building`: reused in subsequent workflow blocks.
- `parametric`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `PROJECT_ROOT`, `idf_file`, `epw_list`, `building`.

In [2]:
PROJECT_ROOT = Path.cwd()

idf_file = "OSM_TestResidentialUnit_v01_onlygeometry_SchNatVent_v2520_high_performance.idf"
epw_list = ["Seville.epw"]

building = ef.get_building(idf_file)
accim.utils.reduce_runtime(
    idf_object=building,
    timesteps=2,
    runperiod_begin_month=1,
    runperiod_begin_day_of_month=1,
    runperiod_end_month=12,
    runperiod_end_day_of_month=31,
)
building.idfobjects["SimulationControl"][0].Run_Simulation_for_Sizing_Periods = "No"

parametric = ParametricSimulation(
	building=building,
	parameters_type="accim custom model",
	epws=epw_list,
	output_freqs=["hourly"],
)


Solar distribution has been set to MinimalShadowing.
Shading Calculation Update Frequency was previously set to 20 days, and it has been modified to 20 days.
Maximum Figures in Shadow Overlap Calculations was previously set to 15000, and it has been modified to 200.
Number of Timesteps per Hour was previously set to 6, and it has been modified to 2.

--------------------------------------------------------
Adaptive-Comfort-Control-Implemented Model (ACCIM) v0.7.8
--------------------------------------------------------

This tool allows to apply adaptive setpoint temperatures. 
For further information, please read the documentation: 
https://accim.readthedocs.io/en/master/
For a visual understanding of the tool, please visit the following jupyter notebooks:
-    Using addAccis() to apply adaptive setpoint temperatures
https://accim.readthedocs.io/en/master/jupyter_notebooks/addAccis/using_addAccis.html
-    Using rename_epw_files() to rename the EPWs for proper data analysis after simu

Air distribution effectiveness (always 1) Schedule has been added
VRF Heating Cooling (Northern Hemisphere) Schedule has been added
DefaultFanEffRatioCurve Curve:Cubic Object has been added
VRFTUCoolCapFT Curve:Cubic Object has been added
VRFTUHeatCapFT Curve:Cubic Object has been added
VRFCoolCapFTBoundary Curve:Cubic Object has been added
VRFCoolEIRFTBoundary Curve:Cubic Object has been added
CoolingEIRLowPLR Curve:Cubic Object has been added
VRFHeatCapFTBoundary Curve:Cubic Object has been added
VRFHeatEIRFTBoundary Curve:Cubic Object has been added
HeatingEIRLowPLR Curve:Cubic Object has been added
DefaultFanPowerRatioCurve Curve:Exponent Object has been added
DXHtgCoilDefrostEIRFT Curve:Biquadratic Object has been added
VRFCoolCapFT Curve:Biquadratic Object has been added
VRFCoolCapFTHi Curve:Biquadratic Object has been added
VRFCoolEIRFT Curve:Biquadratic Object has been added
VRFCoolEIRFTHi Curve:Biquadratic Object has been added
VRFHeatCapFT Curve:Biquadratic Object has been ad

Added - 46 per-ComfStand SetAST subprograms
Added - SetAST Program (base program that calls SetAST_Master)
Added - SetASTnoTol Program
Added - CountHoursNoApp_Floor_1_Residential_Living_Occupants Program
Added - SetGeoVarFloor_1_Residential_Living_Occupants Program
Added - CountHoursNoApp_Floor_2_Residential_Living_Occupants Program
Added - SetGeoVarFloor_2_Residential_Living_Occupants Program
Added - SetInputData Program
Added - SetVOFinputData Program
Added - SetVST Program
Added - ApplyAST_Floor_1_Residential_Living_Occupants Program
Added - ApplyAST_Floor_2_Residential_Living_Occupants Program
Added - Comfort Temperature Output Variable
Added - Adaptive Cooling Setpoint Temperature Output Variable
Added - Adaptive Heating Setpoint Temperature Output Variable
Added - Adaptive Cooling Setpoint Temperature_No Tolerance Output Variable
Added - Adaptive Heating Setpoint Temperature_No Tolerance Output Variable
Added - z_test_ComfStand Output Variable
Added - z_test_ACSTaul Output Variab

## Output preflight: audit output objects already present in the IDF

Inspect existing output objects in the IDF and optionally autocorrect duplicates.

This cell audits output objects already present in the IDF and applies optional cleanup/autocorrection steps. It helps avoid duplicate or conflicting output definitions.

Key arguments used in this cell and why they matter:
- `warn`: defines this setting for the current call.

Variables created in this cell:
- `scan_before`: reused in subsequent workflow blocks.
- `autocorrect_report`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Review audit/autocorrect reports for duplicates or malformed definitions.
- Confirm the IDF is ready for output preflight without redundant objects.

In [3]:
scan_before = parametric.scan_output_objects()
print("Current output meters in IDF:", len(scan_before.get("meters", [])))
print("Current output vars in IDF:", len(scan_before.get("variables", [])))

autocorrect_report = parametric.autocorrect_output_duplicates(warn=True)
print("Autocorrect report keys:", list(autocorrect_report.keys()))


Current output meters in IDF: 8
Current output vars in IDF: 113
Autocorrect report keys: ['idf_scope', 'buildings', 'removed_variables', 'removed_meters']


## Output preflight: discover available meter and variable outputs

Inspect available meters and variables before selecting valid outputs.

Before selecting outputs, we inspect what meters and variables are currently available. That discovery step reduces mistakes when requesting objective or analysis outputs.

Key arguments used in this cell and why they matter:
- `reduce_sim_time`: enables a fast scan mode when discovering outputs.
- `prefer`: preferred backend for output discovery.

Variables created in this cell:
- `df_meters_available`: reused in subsequent workflow blocks.
- `df_vars_available`: reused in subsequent workflow blocks.
- `outputs_meta`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Check that the discovered meter/variable tables are not unexpectedly empty.
- Confirm the reported discovery source matches your preferred backend.

In [4]:
available_outputs = parametric.discover_available_outputs(
	reduce_sim_time=True,
	prefer="testsimeplus",
)
df_meters_available = available_outputs["meters"]
df_vars_available = available_outputs["variables"]
outputs_meta = available_outputs["meta"]
print("Discovery source:", outputs_meta.get("source"))
print("Meters available:", len(df_meters_available))
print("Variables available:", len(df_vars_available))


Solar distribution is already set to MinimalShadowing, therefore no action has been performed.
Shading Calculation Update Frequency was previously set to 20 days, and it has been modified to 20 days.
Maximum Figures in Shadow Overlap Calculations was previously set to 200, and it has been modified to 200.
Number of Timesteps per Hour was previously set to 2, and it has been modified to 2.


Discovery source: testsimeplus
Meters available: 6
Variables available: 144


## Output preflight: select objective and analysis outputs

Select target outputs and validate them before applying changes to the model.

This section applies explicit output selection and preflight validation. Validating before execution keeps result schemas stable across runs.

Key arguments used in this cell and why they matter:
- `meters`: energy meters to include as outputs.
- `variables`: output variables to include by name or pattern.
- `match`: matching rule used to select outputs.
- `on_missing`: behavior when a requested output is missing.
- `suggest`: shows suggestions when no exact match is found.

Variables created in this cell:
- `df_meters_sel`: reused in subsequent workflow blocks.
- `df_vars_sel`: reused in subsequent workflow blocks.
- `report`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm the selected tables include all objective outputs you plan to track.
- Verify preflight did not report blocking missing outputs.

In [5]:
selection = parametric.select_outputs(
	meters=["Heating:Electricity", "Cooling:Electricity"],
	variables=["Zone Operative Temperature", "Running Average Outdoor Air Temperature"],
	match="contains",
	on_missing="warn",
	suggest=True,
)
df_meters_sel = selection["meters"]
df_vars_sel = selection["variables"]
report = selection["report"]

print("Selected meters:", len(df_meters_sel))
print("Selected variables:", len(df_vars_sel))
print("Missing:", report.get("missing", {}))


Selected meters: 2
Selected variables: 6
Missing: {'meters': [], 'variables': []}


Select target outputs and validate them before applying changes to the model.

This section applies explicit output selection and preflight validation. Validating before execution keeps result schemas stable across runs.

Key arguments used in this cell and why they matter:
- `df_meters_sel`: table of selected meters to apply to the IDF.
- `df_vars_sel`: table of selected variables to apply to the IDF.
- `clean_mode`: cleanup mode applied before adding outputs.
- `validate_before_apply`: whether to validate before editing the IDF.
- `validate_after_apply`: whether to validate after editing the IDF.
- `on_missing`: behavior when a requested output is missing.
- `df_output_meter`: final meter table used by the simulation.
- `df_output_variable`: final variable table used by the simulation.

Variables created in this cell:
- `preflight_report`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm the selected tables include all objective outputs you plan to track.
- Verify preflight did not report blocking missing outputs.

In [6]:
preflight_report = parametric.apply_outputs_preflight(
	df_meters_sel=df_meters_sel,
	df_vars_sel=df_vars_sel,
	clean_mode="all",
	validate_before_apply=False,
	validate_after_apply=True,
	on_missing="warn",
)

parametric.set_output_readers(
	df_output_meter=df_meters_sel,
	df_output_variable=df_vars_sel,
)

print("Preflight verification keys:", list(preflight_report.get("verification", {}).keys()))
print("Simulation output readers configured:", len(parametric.sim_outputs))


Preflight verification keys: ['vars', 'meters']
Simulation output readers configured: 8


## Output preflight: verify outputs currently stored in the IDF

Inspect output meter/variable objects currently stored in the IDF.

This section verifies the output objects currently stored in the edited IDF. It is a direct confirmation that preflight and apply steps produced the intended schema.

Variables created in this cell:
- `df_meters_in_idf`: reused in subsequent workflow blocks.
- `df_vars_in_idf`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Check that output counts in the IDF match your selected configuration.
- Inspect a sample of names to confirm no unexpected replacements occurred.

In [7]:
df_meters_in_idf = parametric.get_output_meters_df_from_idf()
df_vars_in_idf = parametric.get_output_variables_df_from_idf()

print("Output meters in IDF after apply:", len(df_meters_in_idf))
print("Output variables in IDF after apply:", len(df_vars_in_idf))
print(df_meters_in_idf.head())


Output meters in IDF after apply: 2
Output variables in IDF after apply: 6
              key_name frequency
0  COOLING:ELECTRICITY    Hourly
1  HEATING:ELECTRICITY    Hourly
